# 🍋 Lemonade Hackathon Workshop

**Fully local AI — no cloud keys, no rate limits, runs on your own GPU/NPU.**

This notebook walks you through every major capability of [Lemonade Server](https://lemonade-server.ai) in ~1 hour:

| Section | What you'll build |
|---------|-------------------|
| 0. Setup | Verify Lemonade is running, pull starter models |
| 1. LLM Chat | Chat with a local model via the OpenAI SDK |
| 2. Speech-to-Text | Transcribe audio files locally with Whisper |
| 3. Image Generation | Generate images with Stable Diffusion |
| 4. Text-to-Speech | Convert text to speech with Kokoro |
| 5. Agentic Loop | LLM that picks and calls multimodal tools |
| 6. MCP Gateway | Inspect the built-in MCP server |
| 7. Hackathon Starter | Your blank canvas — start hacking here |

> **Prerequisite:** Install Lemonade Server from [lemonade-server.ai](https://lemonade-server.ai/install_options.html). The server starts automatically after install.

---
## Section 0 — Setup

Install Python dependencies and confirm Lemonade Server is reachable.

In [ ]:
# Install dependencies (run once)
%pip install openai requests --quiet

In [ ]:
import requests

LEMONADE_URL = "http://localhost:13305"

resp = requests.get(f"{LEMONADE_URL}/health", timeout=5)
health = resp.json()
print("✅ Lemonade Server is running")
print(f"   Version      : {health.get('version', 'unknown')}")
print(f"   WebSocket port: {health.get('websocket_port', 'n/a')}")

In [ ]:
# See what models are already downloaded
resp = requests.get(f"{LEMONADE_URL}/v1/models", timeout=10)
models = resp.json().get("data", [])

if models:
    print("Downloaded models:")
    for m in models:
        print(f"  • {m['id']}")
else:
    print("No models downloaded yet — run the pull cells below.")

In [ ]:
# Pull the models used in this workshop
# Run from a terminal (may take a few minutes depending on your connection):
#
#   lemonade pull Qwen3-1.7B-GGUF        # ~1 GB  — LLM chat + tool calling
#   lemonade pull Whisper-Tiny           # ~75 MB — speech-to-text
#   lemonade pull SD-Turbo               # ~1.9 GB — image generation
#   lemonade pull kokoro-v1              # ~350 MB — text-to-speech
#
# Or download LMX-Omni-5.5B-Lite from the desktop Model Manager
# (bundles all four above into one omni model)

import subprocess, sys

WORKSHOP_MODELS = [
    "Qwen3-1.7B-GGUF",
    "Whisper-Tiny",
    "SD-Turbo",
    "kokoro-v1",
]

for model in WORKSHOP_MODELS:
    print(f"Pulling {model}...")
    result = subprocess.run(
        ["lemonade", "pull", model],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {model} ready")
    else:
        print(f"  ⚠️  {model}: {result.stderr.strip() or result.stdout.strip()}")

---
## Section 1 — LLM Chat

Lemonade exposes the **OpenAI-compatible** `/v1/chat/completions` endpoint.  
The only thing that changes from cloud OpenAI is `base_url`.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f"{LEMONADE_URL}/v1",
    api_key="lemonade",  # required by SDK but not enforced by Lemonade
)

LLM_MODEL = "Qwen3-1.7B-GGUF"

resp = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": "You are a creative assistant for a lemonade hackathon."},
        {"role": "user",   "content": "Write a 2-sentence pitch for an AI-powered lemonade stand app."},
    ],
    max_tokens=150,
)

print(resp.choices[0].message.content)

In [ ]:
# Streaming response — tokens appear as they're generated
stream = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "List 5 fun hackathon project ideas using local AI."}],
    max_tokens=300,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()  # newline at end

---
## Section 2 — Speech-to-Text

Lemonade runs [Whisper](https://github.com/openai/whisper) locally via whisper.cpp.  
The endpoint is OpenAI-compatible: `POST /v1/audio/transcriptions`.

> **Real-time mic transcription:** Run `python examples/realtime_transcription.py` in a terminal tab — Jupyter can't stream live mic audio cleanly.

In [ ]:
import urllib.request, os

# Download a short sample WAV for the demo
SAMPLE_WAV = "sample.wav"
if not os.path.exists(SAMPLE_WAV):
    sample_url = "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3"
    # Use any WAV/MP3 you have locally instead:
    print("⚠️  Place a WAV or MP3 file named 'sample.wav' next to this notebook, then re-run.")
    print("   Or record one with: python examples/realtime_transcription.py")
else:
    print(f"Using: {SAMPLE_WAV}")

In [ ]:
WHISPER_MODEL = "Whisper-Tiny"
AUDIO_FILE    = "sample.wav"  # swap for any WAV/MP3 on your machine

if os.path.exists(AUDIO_FILE):
    with open(AUDIO_FILE, "rb") as f:
        transcript = client.audio.transcriptions.create(
            model=WHISPER_MODEL,
            file=f,
            response_format="text",
        )
    print("Transcript:")
    print(transcript)
else:
    print("⚠️  No audio file found. Add 'sample.wav' next to this notebook to try transcription.")

---
## Section 3 — Image Generation

Lemonade runs Stable Diffusion locally via stable-diffusion.cpp.  
The endpoint is OpenAI-compatible: `POST /v1/images/generations`.

In [ ]:
import base64
from pathlib import Path
from IPython.display import Image, display

IMAGE_MODEL = "SD-Turbo"
IMAGE_PROMPT = "a lemon-shaped spaceship launching from a sunny meadow, digital art"

print(f"Generating image... (first run downloads {IMAGE_MODEL} if not already pulled)")

resp = client.images.generate(
    model=IMAGE_MODEL,
    prompt=IMAGE_PROMPT,
    size="512x512",
    n=1,
    response_format="b64_json",
    extra_body={"steps": 4, "cfg_scale": 1.0},  # SD-Turbo sweet spot
)

img_bytes = base64.b64decode(resp.data[0].b64_json)
out_path = Path("generated_image.png")
out_path.write_bytes(img_bytes)

print(f"✅ Saved to {out_path.absolute()}")
display(Image(filename=str(out_path)))

---
## Section 4 — Text-to-Speech

Lemonade runs [Kokoro](https://github.com/hexgrad/kokoro) locally.  
The endpoint is OpenAI-compatible: `POST /v1/audio/speech`.

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

TTS_MODEL = "kokoro-v1"
TTS_TEXT  = "Welcome to the Lemonade Hackathon! Let's build something amazing with local AI."

print("Generating speech...")

resp = client.audio.speech.create(
    model=TTS_MODEL,
    voice="af_heart",  # available voices: af_heart, af_bella, am_adam, ...
    input=TTS_TEXT,
)

audio_path = Path("generated_speech.wav")
resp.write_to_file(str(audio_path))

print(f"✅ Saved to {audio_path.absolute()}")
display(Audio(filename=str(audio_path), autoplay=False))

---
## Section 5 — Agentic Loop

The real power of local AI: an **LLM that decides which tools to call**, and Python that executes them.

This is the [OmniRouter pattern](../docs/dev/lemonade-omni.md) — the same one used by Lemonade Arcade to generate playable games from a prompt.

```
User prompt
    → LLM (Qwen3) decides: generate_image? text_to_speech?
    → Python calls /v1/images/generations or /v1/audio/speech
    → Result fed back to LLM
    → Loop until LLM says "done"
```

In [ ]:
import json

# Tool schemas — tell the LLM what tools are available
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "generate_image",
            "description": "Generate an image from a text description.",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {"type": "string", "description": "Detailed image description"},
                },
                "required": ["prompt"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "text_to_speech",
            "description": "Convert text to spoken audio.",
            "parameters": {
                "type": "object",
                "properties": {
                    "input": {"type": "string", "description": "Text to speak aloud"},
                },
                "required": ["input"],
            },
        },
    },
]


def execute_tool(tool_call):
    """Execute a tool call returned by the LLM."""
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    if name == "generate_image":
        resp = client.images.generate(
            model=IMAGE_MODEL,
            prompt=args["prompt"],
            size="512x512",
            n=1,
            response_format="b64_json",
            extra_body={"steps": 4, "cfg_scale": 1.0},
        )
        img_bytes = base64.b64decode(resp.data[0].b64_json)
        path = Path("agent_image.png")
        path.write_bytes(img_bytes)
        print(f"  [tool] generate_image → saved to {path.name}")
        display(Image(filename=str(path)))
        return f"Image generated and saved to {path.name}."

    if name == "text_to_speech":
        resp = client.audio.speech.create(
            model=TTS_MODEL, voice="af_heart", input=args["input"]
        )
        path = Path("agent_speech.wav")
        resp.write_to_file(str(path))
        print(f"  [tool] text_to_speech → saved to {path.name}")
        display(Audio(filename=str(path), autoplay=False))
        return f"Audio saved to {path.name}."

    return f"Unknown tool: {name}"


def run_agent(user_prompt, max_iterations=4):
    """Run the LLM tool-calling loop."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant with tools to generate images and speech. Use them when the user asks."},
        {"role": "user",   "content": user_prompt},
    ]

    for i in range(max_iterations):
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            tools=TOOLS,
            max_tokens=300,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            print(f"\nAssistant: {msg.content}")
            break

        messages.append(msg)
        for tc in msg.tool_calls:
            result = execute_tool(tc)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    else:
        print("(max iterations reached)")


# Try it — change the prompt to see different tools get picked
run_agent("Generate an image of a lemonade stand on the moon, then say a one-line description of it aloud.")

### Shortcut: Omni Model (server-side orchestration)

If you've downloaded `LMX-Omni-5.5B-Lite`, the server runs the agentic loop internally.  
Your client just sends a chat message and gets back text + image + audio.

In [ ]:
# Check if the Omni model is available
resp = requests.get(f"{LEMONADE_URL}/v1/models?show_all=true", timeout=10)
all_models = {m["id"] for m in resp.json().get("data", [])}

OMNI_MODEL = "LMX-Omni-5.5B-Lite"

if OMNI_MODEL in all_models:
    omni_resp = client.chat.completions.create(
        model=OMNI_MODEL,
        messages=[{"role": "user", "content": "Generate a lemon car image and describe it."}],
        max_tokens=300,
    )
    print(omni_resp.choices[0].message.content)
else:
    print(f"'{OMNI_MODEL}' not downloaded.")
    print("Open the Lemonade desktop app → Model Manager → Lemonade → LMX-Omni-5.5B-Lite → Download")
    print("Or pull components individually (already done in Section 0).")

---
## Section 6 — MCP Gateway

Lemonade has a built-in [Model Context Protocol](../docs/api/mcp.md) server at `POST /mcp`.  
Any MCP-compatible client — **Claude Code, Cursor, GitHub Copilot, n8n** — can call your local GPU as tools.

5 tools are exposed:
- `lemonade_list_models` — discover what's available
- `lemonade_chat` — chat with any local LLM
- `lemonade_transcribe_audio` — transcribe a local audio file
- `lemonade_generate_image` — generate an image to disk
- `lemonade_omni` — one-shot multimodal turn (LLM + image + TTS)

In [ ]:
MCP_URL = f"{LEMONADE_URL}/mcp"
HEADERS = {"Content-Type": "application/json"}

# Step 1: initialize
init_resp = requests.post(MCP_URL, headers=HEADERS, json={
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {"protocolVersion": "2025-06-18", "capabilities": {}}
})
print("Initialize:", init_resp.json().get("result", {}).get("serverInfo", {}))

In [ ]:
# Step 2: list available tools
tools_resp = requests.post(MCP_URL, headers=HEADERS, json={
    "jsonrpc": "2.0", "id": 2, "method": "tools/list"
})
tools = tools_resp.json().get("result", {}).get("tools", [])
for t in tools:
    print(f"  🔧 {t['name']}: {t['description'][:80]}")

In [ ]:
# Step 3: call lemonade_chat directly via MCP
chat_resp = requests.post(MCP_URL, headers=HEADERS, json={
    "jsonrpc": "2.0", "id": 3, "method": "tools/call",
    "params": {
        "name": "lemonade_chat",
        "arguments": {
            "model": LLM_MODEL,
            "messages": [{"role": "user", "content": "What is Lemonade Server in one sentence?"}],
            "max_tokens": 64,
        }
    }
})
result = chat_resp.json().get("result", {})
for block in result.get("content", []):
    if block.get("type") == "text":
        print(block["text"])

### Connect Claude Code to Lemonade

Run this in your terminal to launch a fully local coding agent:

```bash
# Interactive model picker
lemonade launch claude

# Or specify a model directly (good for hackathon speed)
lemonade launch claude --model Qwen3-1.7B-GGUF --agent-args "--approval-mode never"

# Use a bigger coding model if you have the VRAM
lemonade pull Qwen3-Coder-30B-A3B-GGUF
lemonade launch claude --model Qwen3-Coder-30B-A3B-GGUF
```

Lemonade auto-configures Claude Code to use the local Anthropic-compatible API.

---
## Section 7 — Hackathon Starter

Pick a track and start building. The cells below give you a skeleton for each.

---

### Track A: Voice App
Build a voice assistant, live transcription tool, or accessibility app.  
**Starter:** combine Whisper (STT) → LLM → Kokoro (TTS) in a loop.

In [ ]:
# Track A skeleton: STT → LLM → TTS pipeline
# Replace the audio_file variable with your own recording
# For live mic input, run: python examples/realtime_transcription.py

audio_file = "sample.wav"  # your input audio

if os.path.exists(audio_file):
    # 1. Transcribe
    with open(audio_file, "rb") as f:
        spoken_text = client.audio.transcriptions.create(
            model=WHISPER_MODEL, file=f, response_format="text"
        )
    print(f"User said: {spoken_text}")

    # 2. LLM response
    llm_resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful voice assistant. Keep replies brief."},
            {"role": "user",   "content": spoken_text},
        ],
        max_tokens=100,
    )
    reply = llm_resp.choices[0].message.content
    print(f"Assistant: {reply}")

    # 3. Speak the reply
    tts_resp = client.audio.speech.create(model=TTS_MODEL, voice="af_heart", input=reply)
    tts_resp.write_to_file("voice_reply.wav")
    display(Audio(filename="voice_reply.wav", autoplay=False))
else:
    print("Add an audio file named 'sample.wav' to try the voice pipeline.")

---
### Track B: Image + LLM Pipeline
Build a creative multimodal app — image generation with AI reasoning, storytelling, style transfer, etc.

In [ ]:
# Track B skeleton: LLM writes a prompt → image gen → LLM describes it

# Step 1: ask the LLM to design an image prompt
concept = "a futuristic hackathon in a lemon grove"  # change this

prompt_resp = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": f"Write a vivid 1-sentence image generation prompt for: {concept}"}],
    max_tokens=80,
)
img_prompt = prompt_resp.choices[0].message.content.strip()
print(f"Image prompt: {img_prompt}")

# Step 2: generate the image
img_resp = client.images.generate(
    model=IMAGE_MODEL, prompt=img_prompt, size="512x512", n=1,
    response_format="b64_json", extra_body={"steps": 4, "cfg_scale": 1.0},
)
img_bytes = base64.b64decode(img_resp.data[0].b64_json)
Path("track_b_image.png").write_bytes(img_bytes)
display(Image(filename="track_b_image.png"))

# TODO: add your own logic here

---
### Track C: Agentic Workflow
Build an agent that orchestrates multiple local AI tools — RAG, code generation, data analysis, etc.

In [ ]:
# Track C skeleton: extend the agentic loop with your own tools
# Add a new tool to TOOLS above and a new branch in execute_tool()

MY_TOOLS = TOOLS + [
    {
        "type": "function",
        "function": {
            "name": "my_custom_tool",
            "description": "Describe what your tool does here.",
            "parameters": {
                "type": "object",
                "properties": {
                    "input": {"type": "string", "description": "Input to your tool"},
                },
                "required": ["input"],
            },
        },
    }
]

def my_custom_tool_handler(args):
    # TODO: implement your tool here
    return f"Got input: {args['input']}"

def execute_tool_extended(tool_call):
    if tool_call.function.name == "my_custom_tool":
        args = json.loads(tool_call.function.arguments)
        return my_custom_tool_handler(args)
    return execute_tool(tool_call)  # fall back to built-in tools

# run_agent("Your prompt here", tools=MY_TOOLS)  # wire up execute_tool_extended above

---
### Track D: Coding Agent
Use `lemonade launch claude` to build a game, app, or tool with a fully local AI pair programmer.

Run in your terminal:

```bash
# Small model — fast, good for quick iteration
lemonade launch claude --model Qwen3-1.7B-GGUF --agent-args "--approval-mode never"

# Bigger coding model — better quality
lemonade pull Qwen3-Coder-30B-A3B-GGUF
lemonade launch claude --model Qwen3-Coder-30B-A3B-GGUF
```

**Inspiration — Lemonade Arcade:**  
Type a prompt like `"space invaders with exploding bullets"` and get a playable Python game.  
Source: https://github.com/lemonade-sdk/infinity-arcade

**Inspiration — OpenHands:**  
A full software coding agent that browses, edits, and runs code in a Docker sandbox.  
See `docs/integrations/open-hands.md` for setup.

In [ ]:
# Track D: use the Anthropic-compatible API directly from Python
# (same endpoint lemonade launch claude uses under the hood)

coding_resp = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": "You are an expert Python game developer. Write complete, runnable code."},
        {"role": "user",   "content": "Write a minimal Snake game in Python using pygame. Just the code, no explanation."},
    ],
    max_tokens=1000,
)
print(coding_resp.choices[0].message.content)

---
## Quick Reference

| What | Command / endpoint |
|------|--------------------|
| Server status | `lemonade status` or `GET /health` |
| List models | `lemonade list` or `GET /v1/models` |
| Pull a model | `lemonade pull <name>` |
| Chat | `POST /v1/chat/completions` |
| Transcribe audio | `POST /v1/audio/transcriptions` |
| Generate image | `POST /v1/images/generations` |
| Text-to-speech | `POST /v1/audio/speech` |
| Embeddings | `POST /v1/embeddings` |
| MCP tools | `POST /mcp` |
| Launch coding agent | `lemonade launch claude --model <name>` |
| Real-time mic STT | `python examples/realtime_transcription.py` |
| Multi-model debate | open `examples/llm-debate.html` in browser |

**Default port:** `13305`  
**Docs:** https://lemonade-server.ai/docs/  
**Discord:** https://discord.gg/5xXzkMu8Zk